# Quick Fix - Core Detection

Simplified approach focusing on what works for core tray images.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

In [ ]:
# Load image
image_path = 'path/to/your/core_image.jpg'
image = cv2.imread(image_path)
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

print(f"Image shape: {image.shape}")
print(f"Gray min: {gray.min()}, max: {gray.max()}, mean: {gray.mean():.1f}")

plt.figure(figsize=(15, 10))
plt.imshow(image_rgb)
plt.title('Original Image')
plt.axis('off')
plt.show()

## Problem Diagnosis

Let's see what the binary threshold actually looks like.

In [ ]:
# Test different threshold values
thresholds = [80, 100, 120, 140, 160]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

axes[0].imshow(gray, cmap='gray')
axes[0].set_title('Original Grayscale')
axes[0].axis('off')

for i, thresh_val in enumerate(thresholds):
    _, binary = cv2.threshold(gray, thresh_val, 255, cv2.THRESH_BINARY)
    axes[i+1].imshow(binary, cmap='gray')
    axes[i+1].set_title(f'Threshold = {thresh_val}')
    axes[i+1].axis('off')
    
    # Count white pixels
    white_ratio = np.sum(binary == 255) / binary.size
    axes[i+1].text(10, 30, f'{white_ratio*100:.1f}% white', 
                   color='red', fontsize=10, fontweight='bold',
                   bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.show()

## Try Otsu's Automatic Thresholding

In [ ]:
# Otsu's method automatically finds the best threshold
otsu_thresh, binary_otsu = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

print(f"Otsu's automatic threshold: {otsu_thresh:.1f}")

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes[0].imshow(gray, cmap='gray')
axes[0].set_title('Grayscale')
axes[0].axis('off')

axes[1].imshow(binary_otsu, cmap='gray')
axes[1].set_title(f"Otsu Binary (threshold={otsu_thresh:.1f})")
axes[1].axis('off')

plt.tight_layout()
plt.show()

## Check What Morphology Does

In [ ]:
# Use Otsu threshold
_, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# Apply morphological operations step by step
kernel_small = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
kernel_medium = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
kernel_large = cv2.getStructuringElement(cv2.MORPH_RECT, (7, 7))

# Opening removes small white noise
opening = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel_small, iterations=1)

# Closing fills small black holes
closing = cv2.morphologyEx(opening, cv2.MORPH_CLOSE, kernel_medium, iterations=2)

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

axes[0, 0].imshow(binary, cmap='gray')
axes[0, 0].set_title('1. Binary (Otsu)')
axes[0, 0].axis('off')

axes[0, 1].imshow(opening, cmap='gray')
axes[0, 1].set_title('2. After Opening (remove noise)')
axes[0, 1].axis('off')

axes[1, 0].imshow(closing, cmap='gray')
axes[1, 0].set_title('3. After Closing (fill holes)')
axes[1, 0].axis('off')

axes[1, 1].imshow(image_rgb)
axes[1, 1].set_title('Original for reference')
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

## Find ALL Contours (Before Filtering)

In [ ]:
# Find contours on the cleaned binary
contours, _ = cv2.findContours(closing, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

print(f"Total contours found: {len(contours)}")

# Analyze ALL contours
if len(contours) > 0:
    areas = [cv2.contourArea(c) for c in contours]
    print(f"\nContour areas:")
    print(f"  Min: {min(areas):.0f}")
    print(f"  Max: {max(areas):.0f}")
    print(f"  Mean: {np.mean(areas):.0f}")
    print(f"  Median: {np.median(areas):.0f}")
    
    # Show distribution
    plt.figure(figsize=(12, 4))
    plt.hist(areas, bins=50, edgecolor='black')
    plt.axvline(2000, color='red', linestyle='--', label='min_area=2000')
    plt.xlabel('Contour Area (pixels)')
    plt.ylabel('Count')
    plt.title('Distribution of Contour Areas')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
    
    # Show aspect ratios
    aspect_ratios = []
    for c in contours:
        x, y, w, h = cv2.boundingRect(c)
        if h > 0:
            aspect_ratios.append(w / h)
    
    plt.figure(figsize=(12, 4))
    plt.hist(aspect_ratios, bins=50, edgecolor='black')
    plt.axvline(1.2, color='red', linestyle='--', label='min_aspect=1.2')
    plt.axvline(30, color='red', linestyle='--', label='max_aspect=30')
    plt.xlabel('Aspect Ratio (width/height)')
    plt.ylabel('Count')
    plt.title('Distribution of Aspect Ratios')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.xlim(0, 50)
    plt.show()
else:
    print("\n⚠️ NO CONTOURS FOUND! The binary image might be all black or all white.")

In [ ]:
# Draw ALL contours on the image
vis_all = image_rgb.copy()
cv2.drawContours(vis_all, contours, -1, (0, 255, 0), 2)

plt.figure(figsize=(15, 10))
plt.imshow(vis_all)
plt.title(f'ALL {len(contours)} Contours (Before Filtering)')
plt.axis('off')
plt.show()

## Show Top 20 Largest Contours with Details

In [ ]:
# Get detailed info on largest contours
contour_data = []
for i, c in enumerate(contours):
    area = cv2.contourArea(c)
    x, y, w, h = cv2.boundingRect(c)
    aspect = w / h if h > 0 else 0
    contour_data.append({
        'id': i,
        'area': area,
        'x': x, 'y': y, 'w': w, 'h': h,
        'aspect': aspect
    })

# Sort by area
contour_data.sort(key=lambda x: x['area'], reverse=True)

print("Top 20 largest contours:")
print(f"{'ID':<5} {'Area':<10} {'Width':<7} {'Height':<7} {'Aspect':<7} {'Pass Filter?'}")
print("-" * 60)

for data in contour_data[:20]:
    # Check if passes filters
    passes = (
        data['area'] >= 2000 and 
        data['area'] <= 500000 and
        data['aspect'] >= 1.2 and 
        data['aspect'] <= 30
    )
    status = "✓ YES" if passes else "✗ NO"
    
    print(f"{data['id']:<5} {data['area']:<10.0f} {data['w']:<7} {data['h']:<7} {data['aspect']:<7.2f} {status}")
    
    if not passes:
        reasons = []
        if data['area'] < 2000:
            reasons.append(f"area too small ({data['area']:.0f} < 2000)")
        if data['area'] > 500000:
            reasons.append(f"area too large ({data['area']:.0f} > 500000)")
        if data['aspect'] < 1.2:
            reasons.append(f"aspect too small ({data['aspect']:.2f} < 1.2)")
        if data['aspect'] > 30:
            reasons.append(f"aspect too large ({data['aspect']:.2f} > 30)")
        print(f"      Reason: {', '.join(reasons)}")

## Visualize Top Contours with Numbers

In [ ]:
# Show top 30 largest contours with labels
fig, ax = plt.subplots(figsize=(15, 10))
ax.imshow(image_rgb)

for i, data in enumerate(contour_data[:30]):
    x, y, w, h = data['x'], data['y'], data['w'], data['h']
    
    # Check if passes filters
    passes = (
        data['area'] >= 2000 and 
        data['area'] <= 500000 and
        data['aspect'] >= 1.2 and 
        data['aspect'] <= 30
    )
    
    color = 'lime' if passes else 'red'
    
    rect = Rectangle((x, y), w, h, linewidth=2, edgecolor=color, facecolor='none')
    ax.add_patch(rect)
    ax.text(x + 5, y + 15, f"{i+1}\n{data['area']:.0f}", 
           color='yellow', fontsize=8, fontweight='bold',
           bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))

ax.set_title('Top 30 Contours (Green=Pass, Red=Fail)')
ax.axis('off')
plt.tight_layout()
plt.show()

## Solution: Adjust Filters Based on Data

Based on the statistics above, adjust these parameters:

In [ ]:
# ADJUST THESE BASED ON THE STATISTICS ABOVE!
MIN_AREA = 1000          # Lower if cores are being filtered out
MAX_AREA = 1000000       # Increase if large cores are filtered out
MIN_ASPECT = 0.5         # Lower to allow more square pieces
MAX_ASPECT = 50          # Increase for very elongated pieces

# Filter contours
valid_boxes = []
for data in contour_data:
    if (MIN_AREA <= data['area'] <= MAX_AREA and
        MIN_ASPECT <= data['aspect'] <= MAX_ASPECT):
        valid_boxes.append((data['x'], data['y'], data['w'], data['h']))

# Sort by position
valid_boxes.sort(key=lambda box: (box[1], box[0]))

print(f"\n✓ Detected {len(valid_boxes)} valid core pieces")
print(f"\nFilter settings:")
print(f"  Area: {MIN_AREA} - {MAX_AREA}")
print(f"  Aspect: {MIN_ASPECT} - {MAX_ASPECT}")

# Visualize
fig, ax = plt.subplots(figsize=(15, 10))
ax.imshow(image_rgb)

for i, (x, y, w, h) in enumerate(valid_boxes):
    rect = Rectangle((x, y), w, h, linewidth=2, edgecolor='lime', facecolor='none')
    ax.add_patch(rect)
    ax.text(x + 5, y + 15, str(i+1), color='yellow', fontsize=10, fontweight='bold',
           bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))

ax.set_title(f'Final Result: {len(valid_boxes)} Core Pieces Detected')
ax.axis('off')
plt.tight_layout()
plt.show()